In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import KMeansSMOTE

In [12]:
df = pd.read_csv(r"C:\Users\ankit\OneDrive\Desktop\Research_Papers\UPI_2024_data.zip")
df.info()                     
df.describe()                 
df.nunique()                   
df.isnull().sum()              

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

transaction id        0
timestamp             0
transaction type      0
merchant_category     0
amount (INR)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64

In [3]:
df['transaction_status'] = df['transaction_status'].map({'SUCCESS': 1, 'FAILED': 0})

age_order = ['18-25', '26-35', '36-45', '46-55', '56+']
df['sender_age_group'] = pd.Categorical(df['sender_age_group'], categories=age_order, ordered=True).codes
df['receiver_age_group'] = pd.Categorical(df['receiver_age_group'], categories=age_order, ordered=True).codes

df['hour_sin'] = np.sin(2*np.pi*df['hour_of_day']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour_of_day']/24)

day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4, 'Saturday':5, 'Sunday':6}
df['day_num'] = df['day_of_week'].map(day_map)
df['day_sin'] = np.sin(2*np.pi*df['day_num']/7)
df['day_cos'] = np.cos(2*np.pi*df['day_num']/7)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['year'] = df['timestamp'].dt.year
df['quarter'] = df['timestamp'].dt.quarter
df['week_of_year'] = df['timestamp'].dt.isocalendar().week.astype(int)

df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

nominal_cols = ['transaction type', 'merchant_category', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

df = df.drop(columns=['transaction id', 'timestamp', 'day_of_week'])

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.shape)
print(df.select_dtypes(include='object').columns)
df.dtypes

(250000, 59)
Index([], dtype='object')


amount (INR)                         int64
transaction_status                   int64
sender_age_group                      int8
receiver_age_group                    int8
fraud_flag                           int64
hour_of_day                          int64
is_weekend                           int64
hour_sin                           float64
hour_cos                           float64
day_num                              int64
day_sin                            float64
day_cos                            float64
month                                int32
day                                  int32
year                                 int32
quarter                              int32
week_of_year                         int64
month_sin                          float64
month_cos                          float64
transaction type_P2M                 int64
transaction type_P2P                 int64
transaction type_Recharge            int64
merchant_category_Entertainment      int64
merchant_ca

In [4]:
df.columns.tolist()  

['amount (INR)',
 'transaction_status',
 'sender_age_group',
 'receiver_age_group',
 'fraud_flag',
 'hour_of_day',
 'is_weekend',
 'hour_sin',
 'hour_cos',
 'day_num',
 'day_sin',
 'day_cos',
 'month',
 'day',
 'year',
 'quarter',
 'week_of_year',
 'month_sin',
 'month_cos',
 'transaction type_P2M',
 'transaction type_P2P',
 'transaction type_Recharge',
 'merchant_category_Entertainment',
 'merchant_category_Food',
 'merchant_category_Fuel',
 'merchant_category_Grocery',
 'merchant_category_Healthcare',
 'merchant_category_Other',
 'merchant_category_Shopping',
 'merchant_category_Transport',
 'merchant_category_Utilities',
 'sender_state_Delhi',
 'sender_state_Gujarat',
 'sender_state_Karnataka',
 'sender_state_Maharashtra',
 'sender_state_Rajasthan',
 'sender_state_Tamil Nadu',
 'sender_state_Telangana',
 'sender_state_Uttar Pradesh',
 'sender_state_West Bengal',
 'sender_bank_HDFC',
 'sender_bank_ICICI',
 'sender_bank_IndusInd',
 'sender_bank_Kotak',
 'sender_bank_PNB',
 'sender_ban

In [5]:
X = df.drop(columns=['fraud_flag'])
y = df['fraud_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set fraud cases:", y_train.sum(), "/", len(y_train))
print("Test set fraud cases:", y_test.sum(), "/", len(y_test), "\n")

Train set fraud cases: 384 / 200000
Test set fraud cases : 96 / 50000 



In [6]:
kmeans_smote = KMeansSMOTE(
    kmeans_estimator=MiniBatchKMeans(n_clusters=20, random_state=42, n_init=10),
    cluster_balance_threshold=0.001,
    k_neighbors=3,
    random_state=42
)

X_train_res, y_train_res = kmeans_smote.fit_resample(X_train, y_train)

print("After K-Means SMOTE, training class balance:")
print(y_train_res.value_counts(), "\n")

After K-Means SMOTE, training class balance:
fraud_flag
1    199627
0    199616
Name: count, dtype: int64 



In [1]:
# clf = DecisionTreeClassifier(random_state=42, max_depth=8)
clf = xgb.XGBClassifier(
    random_state=42, 
    max_depth=2,          
    min_child_weight=5,      
    scale_pos_weight=520, 
    n_estimators=150,        
    learning_rate=0.05,     
    eval_metric='logloss'
)
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

NameError: name 'xgb' is not defined